# Notebook 07: PPO Improvement Analysis and Guided Search

## Course: 22AIE401 — Reinforcement Learning Project
**Topic:** Reinforcement Learning for Drug-Likeness Optimization of Small Molecules  
**Dataset:** ZINC-250K Scaffold-Stratified Benchmark  

### Overview:
In initial benchmark evaluations, PPO Adaptive V1 underperformed Greedy Hill Climbing. This notebook documents:
1. **Diagnostic Audit:** Identifying action space skewness, runaway over-editing, and GNN bond-order blindness.
2. **Algorithmic Solutions (V2):** Edge-Aware GNN encoder and calibrated reward with STOP preservation bonus.
3. **PPO-Guided Local Search Hybrid:** Bridging policy filtering with local lookahead.
4. **Multi-Method Experimental Comparison:** Empirical performance on 100 held-out test molecules.


In [2]:
import os, sys, pandas as pd, numpy as np
from scipy import stats

print("=== 1. Action Space Imbalance Audit ===")
df_actions = pd.read_csv("results/improvement_100/action_audit_report.csv")
print(df_actions.to_string(index=False))


=== 1. Action Space Imbalance Audit ===
action_type  count  min_index  max_index  percentage
       STOP      1          0          0    0.052604
        ADD    228          1        228   11.993688
     REMOVE     38        229        266    1.998948
 SUBSTITUTE    228        267        494   11.993688
CHANGE_BOND   1406        495       1900   73.961073


### Key Action Space Findings:
- `CHANGE_BOND` represents **73.96%** (1,406 / 1,901) of all actions.
- `STOP` represents only **0.052%** (1 / 1,901).
- Random exploration almost never samples STOP ($0.052\%$ chance per step).


In [4]:
print("=== 2. Diagnostics of Policy Rollout & Termination ===")
df_diag = pd.read_csv("results/improvement_100/diagnostics_summary.csv")
print(df_diag.to_string(index=False))


=== 2. Diagnostics of Policy Rollout & Termination ===
             Diagnostic Metric                   Observed Value                         Benchmark Target / Root Cause
        Mean Evaluation Return                           -1.956        Negative returns indicate penalty accumulation
     Mean Evaluation Delta-QED                          -0.2117       Greedy achieves +0.1229; PPO degrades molecules
                  Success Rate                             5.0%      Greedy achieves 65.4%; PPO fails multi-objective
         Mean Final Similarity                            0.460                Target >= 0.40; drifting near boundary
          Mean Number of Edits                        9.65 / 10 Greedy stops at 1.42 edits; PPO over-edits to horizon
   Episodes Terminated by STOP                             0.0%       Agent rarely chooses STOP; runs until truncated
Episodes Terminated by Horizon                            95.0%                  90%+ episodes hit maximum step limit
 

### Root Causes of PPO V1 Underperformance:
1. **Runaway Over-Editing:** STOP frequency was **0.0%**; 95% of episodes dragged on until hitting the 10-step limit. Mean edits = **9.65 / 10** (vs. 1.42 for Greedy). Because molecular edit space is non-monotonic, multi-step drift degrades QED.
2. **GNN Architecture Blindness:** PyG's `GraphConv` layers completely omitted `edge_attr`, preventing the model from perceiving chemical bond orders (single, double, triple, aromatic).
3. **Reward Disincentive on STOP:** Terminating without formal success incurred a $-\text{step\_cost}$ penalty, forcing the agent to gamble on further edits.


In [6]:
print("=== 3. Master Multi-Method Comparison Table (100 Scaffold-Separated Test Molecules) ===")
df_comp = pd.read_csv("results/improvement_100/ppo_variants_comparison.csv")
cols = ["Method", "Mean Delta-QED", "Formal Success Rate", "High-QED Success Rate", "Mean Similarity", "Constraint Violations", "Mean Delta-SAS", "Avg Prop Evals / Mol", "Inference Sec / Mol"]
print(df_comp[cols].to_string(index=False))


=== 3. Master Multi-Method Comparison Table (100 Scaffold-Separated Test Molecules) ===
                           Method  Mean Delta-QED  Formal Success Rate  High-QED Success Rate  Mean Similarity  Constraint Violations  Mean Delta-SAS  Avg Prop Evals / Mol  Inference Sec / Mol
               Random Valid Edits       -0.465420                 0.03                   0.00         0.222040                   0.95        2.006066                  9.51             0.188087
       PPO Baseline (V1 Unshaped)       -0.386570                 0.05                   0.02         0.263520                   0.87        1.254631                  9.53             0.208562
       PPO Adaptive (V1 Original)       -0.205372                 0.06                   0.10         0.460760                   0.46        0.836232                  9.56             0.228456
PPO Adaptive (V2 Edge-Aware Pure)       -0.177081                 0.00                   0.09         0.715198                   0.00       

## 4. Statistical Significance Tests
Paired two-sided Student's t-test and Wilcoxon signed-rank test comparing PPO-Guided Hybrid vs. PPO Adaptive V1 and Greedy Hill Climbing.


In [8]:
df_detail = pd.read_csv("results/improvement_100/final_validation_comparison.csv")
p_v1 = df_detail[df_detail["method"] == "PPO Adaptive (V1 Original)"]["delta_qed"].values
p_hyb = df_detail[df_detail["method"] == "PPO-Guided Search (Hybrid Top-5)"]["delta_qed"].values
p_grd = df_detail[df_detail["method"] == "Greedy Hill Climbing (Oracle)"]["delta_qed"].values

t1, p1 = stats.ttest_rel(p_hyb, p_v1)
w1, pw1 = stats.wilcoxon(p_hyb - p_v1)

t2, p2 = stats.ttest_rel(p_hyb, p_grd)
w2, pw2 = stats.wilcoxon(p_hyb - p_grd)

print("Statistical Test Results:")
print(f"1. PPO Hybrid vs PPO V1 Original: Paired t={t1:.4f} (p={p1:.3e}), Wilcoxon W={w1:.1f} (p={pw1:.3e}) => Statistically significant improvement (p < 0.001)")
print(f"2. PPO Hybrid vs Greedy Oracle: Paired t={t2:.4f} (p={p2:.3e}), Wilcoxon W={w2:.1f} (p={pw2:.3e}) => Greedy achieves higher raw QED, but at 47.2x computational cost.")


Statistical Test Results:
1. PPO Hybrid vs PPO V1 Original: Paired t=10.7891 (p=2.094e-18), Wilcoxon W=281.0 (p=1.204e-14) => Statistically significant improvement (p < 0.001)
2. PPO Hybrid vs Greedy Oracle: Paired t=-13.0593 (p=2.935e-23), Wilcoxon W=100.0 (p=2.582e-16) => Greedy achieves higher raw QED, but at 47.2x computational cost.


## 5. Summary and Conclusions
- **PPO-Guided Search (Hybrid Top-5)** achieves **+0.0183 mean Delta-QED** and **41.0% High-QED success**, reversing the negative drift of pure PPO.
- **Computational Efficiency:** Requires only **5.68 property evaluations per molecule**, compared to **268.22 evaluations** for Greedy Hill Climbing (**47.2x faster**).
- **Lead Identity & Safety:** Preserves starting lead structure with **0.8790 Tanimoto similarity** (vs. 0.6255 for Greedy) and minimal synthetic difficulty increase ($\Delta	ext{SAS} = +0.105$ vs. $+0.383$).
